# Gold — Data Quality Checks (Great Expectations)

This notebook implements an industry-standard **Great Expectations (GX)** validation suite.\
It serves as the final "gatekeeper" for the **Iceland Economic Analytics** pipeline, ensuring that only verified, high-integrity data reaches the Power BI Semantic Model.

### Pipeline Behavior
* **Immediate Halt:** Any failed expectation raises a `ValueError`, halting the Fabric Pipeline before "bad" data is exposed to stakeholders.
* **Observability:** Every execution (Pass/Fail) is logged to the `gold.dq_logs` table, providing a historical audit trail of data health and reliability.

**Runs after:** `gold_fact_monthly_economics`, `gold_dim_date`, `gold_dim_indicator`, `gold_dim_source`, `gold_dim_geography`

In [ ]:
import great_expectations as gx
from pyspark.sql import functions as F
from datetime import datetime

# 1. Load Data
df_fact = spark.read.table("gold.fact_monthly_economics")

# 2. Setup Context & Suite
context = gx.get_context()
suite_name = "iceland_economics_suite"

try:
    context.suites.add(gx.ExpectationSuite(name=suite_name))
except Exception:
    pass # Suite already exists

# 3. Connect the Data
datasource = context.data_sources.add_or_update_spark(name="iceland_spark_datasource")
data_asset = datasource.add_dataframe_asset(name="fact_monthly_economics_asset")

# Build the request using 'options' to pass the Spark DataFrame
batch_request = data_asset.build_batch_request(options={"dataframe": df_fact})

# Get the Validator (the engine for running expectations)
validator = context.get_validator(batch_request=batch_request, expectation_suite_name=suite_name)

# 4. Fetch Dimension Keys for Referential Integrity (RI)
dim_date_keys = [r[0] for r in spark.sql("SELECT date_key FROM gold.dim_date").collect()]
dim_ind_keys  = [r[0] for r in spark.sql("SELECT indicator_key FROM gold.dim_indicator").collect()]
dim_src_keys  = [r[0] for r in spark.sql("SELECT source_key FROM gold.dim_source").collect()]
dim_geo_keys  = [r[0] for r in spark.sql("SELECT geography_key FROM gold.dim_geography").collect()]

In [ ]:
# 1. General Integrity
columns = ["month_key", "date_key", "indicator_key", "source_key", "geography_key", "value"]
for column in columns:
    validator.expect_column_values_to_not_be_null(column)

validator.expect_compound_columns_to_be_unique(["month_key", "indicator_key"])

# 2. Referential Integrity
validator.expect_column_values_to_be_in_set("date_key", dim_date_keys)
validator.expect_column_values_to_be_in_set("indicator_key", dim_ind_keys)
validator.expect_column_values_to_be_in_set("source_key", dim_src_keys)
validator.expect_column_values_to_be_in_set("geography_key", dim_geo_keys)

# 3. Value Ranges (Accuracy)

# Policy Rate (0-30%)
validator.expect_column_values_to_be_between(
    "value", 0, 30, 
    row_condition='indicator_key == "policy_rate"',
    condition_parser="spark"
)

# CPI Inflation (-10 to 100%)
validator.expect_column_values_to_be_between(
    "value", -10, 100, 
    row_condition='indicator_key == "cpi"',
    condition_parser="spark"
)

# FX & OMX (Strictly Positive)
validator.expect_column_values_to_be_between(
    "value", 0, strict_min=True, 
    row_condition='indicator_key IN ("close_eurusd_x", "close_iskusd_x", "close_^omx", "iskeur_close")', 
    condition_parser="spark"
)

# 4. COMPLETENESS: Adjusted for Real-World Lags
latest_month = df_fact.select(F.max("month_key")).collect()[0][0]

# We expect 5-7 indicators. 
# 5 = Exchange Rates + Policy Rate (Fast)
# 6 = Adds CPI (Monthly Lag)
# 7 = Adds GDP (Quarterly Lag)
validator.expect_column_unique_value_count_to_be_between(
    "indicator_key", 
    min_value=5, 
    max_value=7, 
    row_condition=f'month_key == {latest_month}', 
    condition_parser="spark"
)

# 5. Volume Sanity
validator.expect_table_row_count_to_be_between(min_value=7)

In [ ]:
# 1. Execute all checks
results = validator.validate()

# 2. Extract Statistics for Observability
stats = results.statistics
success_rate = stats.get("success_percent", 0)

# 3. Log to Delta table
log_entry = [{
    "run_timestamp": datetime.now(),
    "table_name": "gold.fact_monthly_economics",
    "success_rate": success_rate,
    "status": "PASSED" if results.success else "FAILED"
}]

spark.createDataFrame(log_entry).write.format("delta").mode("append").saveAsTable("gold.dq_logs")

# 4. Critical Halt — Throws error if any expectation failed
if not results.success:
    failed = [r.expectation_config.type for r in results.results if not r.success]
    raise ValueError(f"❌ [DQ FAILURE] {len(failed)} checks failed: {failed}. See gold.dq_logs for details.")

print(f"✅ [DQ PASSED] {success_rate}% success rate across {stats.get('evaluated_expectations')} checks.")